# Linear Models - US Equities Panel

This notebook generates walk-forward validation predictions for the published linear-model
configurations. The visible decisions are the label, configurations, parameter overrides, and
execution tier. Shared code owns fold construction, preprocessing, fitting, recovery, coverage
checks, and registry writes.

The normal path is intentionally short:

1. Open the case-study workspace.
2. Select configurations from the published menu.
3. Run them through the shared fold-major implementation.
4. Inspect the returned catalog rows, coverage, lineage, and artifacts.

To study or extend the implementation, open `case_studies/utils/linear.py`. A new estimator or
preprocessing idea remains ordinary Python and can implement the same request/result contract.

**Learning objectives**

- Express a complete linear-model experiment through visible request parameters.
- Distinguish compatible fold preparation from estimator-specific fitting.
- Validate checkpoint identities, prediction coverage, and downstream catalog rows.

**Book reference**: Chapter 11, Section 11.2 (Regularized Linear Models)

**Prerequisites**: `03_financial_features.py`, `04_model_based_features.py`, and
`05_evaluation.py`.

In [1]:
"""Generate linear-model validation predictions through the shared research interface."""

import os
from pathlib import Path

import polars as pl
import yaml

from case_studies.research import Study, plan_models
from utils.modeling import load_configs
from utils.paths import REPO_ROOT, get_case_study_dir

In [2]:
CASE_STUDY_ID = "us_equities_panel"
PRIMARY_LABEL = ""
CONFIG_NAMES = []
CONFIG_OVERRIDES = {}
DIAGNOSTIC_CONFIG_NAMES = ["ridge_a1.0"]
EXECUTION_TIER = "canonical"
WORKSPACE = "experiments"
MAX_SYMBOLS = 0
TRAIN_SAMPLE_FRAC = 1.0
MAX_FOLDS = 0

## Configure the experiment

`CONFIG_NAMES = []` runs the complete published linear menu. Set it to a visible subset such as
`['ridge_a1.0', 'lasso_a0.001']` for a targeted experiment. `CONFIG_OVERRIDES` changes only the
named estimator parameters, for example `{'ridge_a1.0': {'alpha': 2}}`; the resolved training
specification records every effective default and override. `DIAGNOSTIC_CONFIG_NAMES` declares
the bounded subset used for raw prediction comparisons in the analysis notebook.

Canonical execution uses the complete data and fold protocol. For a reduced pipeline check, set
`EXECUTION_TIER = 'preview'` and declare at least one reduction. Preview identities and artifacts
are isolated from official comparisons and holdout decisions.

In [3]:
case_dir = get_case_study_dir(CASE_STUDY_ID)
setup = yaml.safe_load((case_dir / "config" / "setup.yaml").read_text())
label = PRIMARY_LABEL or setup["labels"]["primary"]

published_configs = load_configs(CASE_STUDY_ID, label, family="linear")
published_names = [str(config["config_name"]) for config in published_configs]
selected_names = list(CONFIG_NAMES) if CONFIG_NAMES else published_names
unknown_names = sorted(set(selected_names) - set(published_names))
unknown_overrides = sorted(set(CONFIG_OVERRIDES) - set(selected_names))
if unknown_names:
    raise ValueError(f"Unknown linear configurations: {unknown_names}")
if unknown_overrides:
    raise ValueError(f"Overrides supplied for unselected configurations: {unknown_overrides}")
if len(selected_names) != len(set(selected_names)):
    raise ValueError("CONFIG_NAMES contains duplicates")
unknown_diagnostics = sorted(set(DIAGNOSTIC_CONFIG_NAMES) - set(selected_names))
if not DIAGNOSTIC_CONFIG_NAMES or unknown_diagnostics:
    raise ValueError(f"Invalid diagnostic configurations: {unknown_diagnostics}")

menu = pl.DataFrame(
    {
        "config_name": [config["config_name"] for config in published_configs],
        "model_class": [config["model_class"] for config in published_configs],
        "published_params": [str(config.get("params") or {}) for config in published_configs],
        "selected": [config["config_name"] in selected_names for config in published_configs],
    }
)
menu

config_name,model_class,published_params,selected
str,str,str,bool
"""ols""","""LinearRegression""","""{}""",true
"""ridge_a0.001""","""Ridge""","""{'alpha': 0.001}""",true
"""ridge_a0.01""","""Ridge""","""{'alpha': 0.01}""",true
"""ridge_a0.1""","""Ridge""","""{'alpha': 0.1}""",true
"""ridge_a1.0""","""Ridge""","""{'alpha': 1.0}""",true
…,…,…,…
"""ridge_a10000000.0""","""Ridge""","""{'alpha': 10000000.0}""",true
"""lasso_a0.01""","""Lasso""","""{'alpha': 0.01, 'max_iter': 10…",true
"""lasso_a0.1""","""Lasso""","""{'alpha': 0.1, 'max_iter': 100…",true


In [4]:
preview_reductions = {}
if MAX_SYMBOLS:
    preview_reductions["max_symbols"] = int(MAX_SYMBOLS)
if TRAIN_SAMPLE_FRAC != 1.0:
    preview_reductions["train_sample_frac"] = float(TRAIN_SAMPLE_FRAC)
if MAX_FOLDS:
    preview_reductions["folds"] = list(range(int(MAX_FOLDS)))

if EXECUTION_TIER == "canonical":
    if preview_reductions:
        raise ValueError("Canonical execution cannot declare preview reductions")
    study = Study.regenerate(CASE_STUDY_ID, release_root=REPO_ROOT)
elif EXECUTION_TIER == "preview":
    if not preview_reductions:
        raise ValueError("Preview execution requires at least one declared reduction")
    study = Study.open(
        CASE_STUDY_ID,
        workspace=Path(os.environ.get("ML4T_OUTPUT_DIR") or WORKSPACE),
        release_root=REPO_ROOT,
    )
else:
    raise ValueError("EXECUTION_TIER must be 'canonical' or 'preview'")

## Build the model requests

Each selected configuration becomes one visible request with its own estimator overrides.

In [5]:
requests = tuple(
    study.model(
        family="linear",
        label=label,
        config_name=config_name,
        overrides=dict(CONFIG_OVERRIDES.get(config_name, {})),
        execution_tier=EXECUTION_TIER,
        preview_reductions=preview_reductions,
    )
    for config_name in selected_names
)

request_table = pl.DataFrame(
    {
        "family": [request.family for request in requests],
        "label": [request.label for request in requests],
        "config_name": [request.config_name for request in requests],
        "overrides": [str(request.overrides) for request in requests],
        "execution_tier": [request.execution_tier.value for request in requests],
        "preview_reductions": [str(request.preview_reductions) for request in requests],
    }
)
request_table

family,label,config_name,overrides,execution_tier,preview_reductions
str,str,str,str,str,str
"""linear""","""fwd_ret_1d""","""ols""","""{}""","""canonical""","""{}"""
"""linear""","""fwd_ret_1d""","""ridge_a0.001""","""{}""","""canonical""","""{}"""
"""linear""","""fwd_ret_1d""","""ridge_a0.01""","""{}""","""canonical""","""{}"""
"""linear""","""fwd_ret_1d""","""ridge_a0.1""","""{}""","""canonical""","""{}"""
"""linear""","""fwd_ret_1d""","""ridge_a1.0""","""{}""","""canonical""","""{}"""
…,…,…,…,…,…
"""linear""","""fwd_ret_1d""","""ridge_a10000000.0""","""{}""","""canonical""","""{}"""
"""linear""","""fwd_ret_1d""","""lasso_a0.01""","""{}""","""canonical""","""{}"""
"""linear""","""fwd_ret_1d""","""lasso_a0.1""","""{}""","""canonical""","""{}"""


## Plan and execute the selected configurations

`plan_models` resolves every training and checkpoint identity before fitting. The canonical
checkpoint population is written first, so a failed configuration remains visible as missing.
The plan materializes the panel once for the compatible request batch. Execution then prepares
one compatible fold at a time for fitting.
Each configuration still receives its own immutable training and prediction identities. Completed
folds are committed incrementally, so a retry reuses valid work and recomputes only incomplete
folds.

In [6]:
plan = plan_models(study, requests=requests)
official_population = None
if EXECUTION_TIER == "canonical":
    official_population = plan.create_population(
        name="us-equities-linear-checkpoints-v1",
    )

planned_population = pl.DataFrame(
    {
        "family": [member.family for member in plan.members],
        "config_name": [member.config_name for member in plan.members],
        "checkpoint_kind": [member.checkpoint_kind for member in plan.members],
        "checkpoint_value": [member.checkpoint_value for member in plan.members],
        "training_hash": [member.training_hash for member in plan.members],
        "prediction_hash": [member.prediction_hash for member in plan.members],
    }
)
planned_population

family,config_name,checkpoint_kind,checkpoint_value,training_hash,prediction_hash
str,str,str,null,str,str
"""linear""","""ols""","""final""",null,"""e0567074e635""","""97a84eee9bfe"""
"""linear""","""ridge_a0.001""","""final""",null,"""c9893f7e4c0c""","""6ccd3bf150be"""
"""linear""","""ridge_a0.01""","""final""",null,"""daf8afb37112""","""fcd529a652a6"""
"""linear""","""ridge_a0.1""","""final""",null,"""c8a7eff4d4ac""","""e35d9b7943bc"""
"""linear""","""ridge_a1.0""","""final""",null,"""ca38fec4f3d4""","""791dfde4a2e5"""
…,…,…,…,…,…
"""linear""","""ridge_a10000000.0""","""final""",null,"""fea6b8598d57""","""59df2d0e56ad"""
"""linear""","""lasso_a0.01""","""final""",null,"""8e8919ea27c9""","""66473a86e9a3"""
"""linear""","""lasso_a0.1""","""final""",null,"""f58ea2f4a346""","""0b472e679983"""


In [7]:
execution = plan.run()

## Inspect the resolved computation

These rows show the feature artifacts, feature count, folds, task, estimator, and immutable
training identity used by each request. Defaults that were not repeated in the parameter cell are
part of the stored specification.

In [8]:
resolved_rows = []
for run in execution.runs:
    spec = run.training.spec()
    computation = spec["computation"]
    feature_artifacts = computation["feature_artifacts"]
    artifact_names = (
        sorted(feature_artifacts)
        if isinstance(feature_artifacts, dict)
        else [str(item) for item in feature_artifacts]
    )
    resolved_rows.append(
        {
            "config_name": spec["config_name"],
            "task": computation["task"]["type"],
            "features": len(computation["feature_names"]),
            "feature_artifacts": artifact_names,
            "folds": computation["expected_prediction_keys"]["n_folds"],
            "estimator": computation["model"]["class"],
            "training_hash": run.training.hash,
        }
    )

resolved_table = pl.DataFrame(resolved_rows).sort("config_name")
resolved_table

config_name,task,features,feature_artifacts,folds,estimator,training_hash
str,str,i64,list[str],i64,str,str
"""enet_a0.01""","""regression""",71,"[""financial"", ""label"", ""model_based""]",16,"""ElasticNet""","""41da576612a9"""
"""enet_a0.1""","""regression""",71,"[""financial"", ""label"", ""model_based""]",16,"""ElasticNet""","""bcb0e6b7e029"""
"""lasso_a0.01""","""regression""",71,"[""financial"", ""label"", ""model_based""]",16,"""Lasso""","""8e8919ea27c9"""
"""lasso_a0.1""","""regression""",71,"[""financial"", ""label"", ""model_based""]",16,"""Lasso""","""f58ea2f4a346"""
"""ols""","""regression""",71,"[""financial"", ""label"", ""model_based""]",16,"""LinearRegression""","""e0567074e635"""
…,…,…,…,…,…,…
"""ridge_a1000.0""","""regression""",71,"[""financial"", ""label"", ""model_based""]",16,"""Ridge""","""b2a210067692"""
"""ridge_a10000.0""","""regression""",71,"[""financial"", ""label"", ""model_based""]",16,"""Ridge""","""72dc4bdfdb57"""
"""ridge_a100000.0""","""regression""",71,"[""financial"", ""label"", ""model_based""]",16,"""Ridge""","""77660a4c1a1e"""


## Validate and inspect the handoff

The returned Polars rows are the downstream interface. A strategy notebook can filter these rows
by human-readable fields and pass the selection directly to `run_backtests`; readers do not need
to copy registry hashes. The hashes remain visible for exact provenance and artifact inspection.

In [9]:
catalog_columns = [
    "family",
    "config_name",
    "label",
    "split",
    "checkpoint_kind",
    "execution_tier",
    "complete",
    "ic_mean",
    "training_hash",
    "prediction_hash",
]
catalog_rows = execution.catalog_rows.select(
    column for column in catalog_columns if column in execution.catalog_rows.columns
).sort("config_name", "prediction_hash")
catalog_rows

family,config_name,label,split,checkpoint_kind,execution_tier,complete,ic_mean,training_hash,prediction_hash
str,str,str,str,str,str,bool,f64,str,str
"""linear""","""enet_a0.01""","""fwd_ret_1d""","""validation""","""final""","""canonical""",true,-0.001786,"""41da576612a9""","""10a1fa2f2a15"""
"""linear""","""enet_a0.1""","""fwd_ret_1d""","""validation""","""final""","""canonical""",true,-0.001786,"""bcb0e6b7e029""","""02511dc4d044"""
"""linear""","""lasso_a0.01""","""fwd_ret_1d""","""validation""","""final""","""canonical""",true,-0.001786,"""8e8919ea27c9""","""66473a86e9a3"""
"""linear""","""lasso_a0.1""","""fwd_ret_1d""","""validation""","""final""","""canonical""",true,-0.001786,"""f58ea2f4a346""","""0b472e679983"""
"""linear""","""ols""","""fwd_ret_1d""","""validation""","""final""","""canonical""",true,0.015907,"""e0567074e635""","""97a84eee9bfe"""
…,…,…,…,…,…,…,…,…,…
"""linear""","""ridge_a1000.0""","""fwd_ret_1d""","""validation""","""final""","""canonical""",true,0.01565,"""b2a210067692""","""9781feec4962"""
"""linear""","""ridge_a10000.0""","""fwd_ret_1d""","""validation""","""final""","""canonical""",true,0.01608,"""72dc4bdfdb57""","""78e91ae5e699"""
"""linear""","""ridge_a100000.0""","""fwd_ret_1d""","""validation""","""final""","""canonical""",true,0.017532,"""77660a4c1a1e""","""b660c7c56b77"""


In [10]:
coverage_rows = []
for run in execution.runs:
    if not run.training.complete:
        raise RuntimeError(f"Incomplete training result: {run.training.hash}")
    for prediction in run.predictions:
        coverage = prediction.coverage()
        if not prediction.complete or coverage is None or coverage["status"] != "complete":
            raise RuntimeError(f"Incomplete prediction result: {prediction.hash}")
        coverage_rows.append(
            {
                "config_name": run.training.spec()["config_name"],
                "training_hash": run.training.hash,
                "prediction_hash": prediction.hash,
                "coverage_status": coverage["status"],
                "expected_rows": coverage["n_expected"],
                "actual_rows": coverage["n_actual"],
                "training_artifacts": len(run.training.artifacts()),
                "prediction_artifacts": len(prediction.artifacts()),
            }
        )

coverage_table = pl.DataFrame(coverage_rows).sort("config_name", "prediction_hash")
if official_population is not None:
    official_population.require_complete()
coverage_table

config_name,training_hash,prediction_hash,coverage_status,expected_rows,actual_rows,training_artifacts,prediction_artifacts
str,str,str,str,i64,i64,i64,i64
"""enet_a0.01""","""41da576612a9""","""10a1fa2f2a15""","""complete""",7170323,7170323,35,1
"""enet_a0.1""","""bcb0e6b7e029""","""02511dc4d044""","""complete""",7170323,7170323,35,1
"""lasso_a0.01""","""8e8919ea27c9""","""66473a86e9a3""","""complete""",7170323,7170323,35,1
"""lasso_a0.1""","""f58ea2f4a346""","""0b472e679983""","""complete""",7170323,7170323,35,1
"""ols""","""e0567074e635""","""97a84eee9bfe""","""complete""",7170323,7170323,35,1
…,…,…,…,…,…,…,…
"""ridge_a1000.0""","""b2a210067692""","""9781feec4962""","""complete""",7170323,7170323,35,1
"""ridge_a10000.0""","""72dc4bdfdb57""","""78e91ae5e699""","""complete""",7170323,7170323,35,1
"""ridge_a100000.0""","""77660a4c1a1e""","""b660c7c56b77""","""complete""",7170323,7170323,35,1


In [11]:
execution_diagnostics = pl.DataFrame(execution.diagnostics)
execution_diagnostics

status,training_hash,cache_hit,reused_folds,fitted_folds,execution_order,compatibility_group,compatibility_group_size,base_fold_preparations,base_fold_preparation_s,candidate_fit_s,preparation_fraction,disk_fold_cache
str,str,bool,list[null],list[i64],str,str,i64,i64,f64,f64,f64,bool
"""completed""","""e0567074e635""",false,[],"[0, 1, … 15]","""fold_major""","""37d32b57f347""",16,16,529.520592,208.71269,0.463148,false
"""completed""","""c9893f7e4c0c""",false,[],"[0, 1, … 15]","""fold_major""","""37d32b57f347""",16,16,529.520592,22.571646,0.463148,false
"""completed""","""daf8afb37112""",false,[],"[0, 1, … 15]","""fold_major""","""37d32b57f347""",16,16,529.520592,22.773072,0.463148,false
"""completed""","""c8a7eff4d4ac""",false,[],"[0, 1, … 15]","""fold_major""","""37d32b57f347""",16,16,529.520592,22.609744,0.463148,false
"""completed""","""ca38fec4f3d4""",false,[],"[0, 1, … 15]","""fold_major""","""37d32b57f347""",16,16,529.520592,22.645651,0.463148,false
…,…,…,…,…,…,…,…,…,…,…,…,…
"""completed""","""fea6b8598d57""",false,[],"[0, 1, … 15]","""fold_major""","""37d32b57f347""",16,16,529.520592,22.578338,0.463148,false
"""completed""","""8e8919ea27c9""",false,[],"[0, 1, … 15]","""fold_major""","""37d32b57f347""",16,16,529.520592,39.064791,0.463148,false
"""completed""","""f58ea2f4a346""",false,[],"[0, 1, … 15]","""fold_major""","""37d32b57f347""",16,16,529.520592,38.92333,0.463148,false


## Freeze the compatible result sets

A canonical default run freezes every returned prediction row under a stable family/label name.
The separately named diagnostic subset is bounded by the visible configuration list above. Preview
and customized canonical requests retain their result rows without publishing an official set.

In [12]:
set_rows = []
is_published_population = (
    EXECUTION_TIER == "canonical" and selected_names == published_names and not CONFIG_OVERRIDES
)
if is_published_population:
    label_name = label.replace("_", "-")
    full_set = study.predictions.freeze(
        execution.catalog_rows,
        name=f"us-equities-{label_name}-linear-v1",
    )
    diagnostic_set = study.predictions.freeze(
        execution.catalog_rows.filter(pl.col("config_name").is_in(DIAGNOSTIC_CONFIG_NAMES)),
        name=f"us-equities-{label_name}-linear-diagnostics-v1",
    )
    set_rows = [
        {
            "role": "backtest population",
            "set_name": full_set.name,
            "members": len(full_set.members),
        },
        {
            "role": "bounded diagnostics",
            "set_name": diagnostic_set.name,
            "members": len(diagnostic_set.members),
        },
    ]
compatible_sets = pl.DataFrame(
    set_rows,
    schema={"role": pl.String, "set_name": pl.String, "members": pl.Int64},
)
compatible_sets

role,set_name,members
str,str,i64
"""backtest population""","""us-equities-fwd-ret-1d-linear-…",16
"""bounded diagnostics""","""us-equities-fwd-ret-1d-linear-…",1


`15_model_analysis.py` reopens the named compatible and diagnostic sets. `16_backtest.py` passes
every full-set catalog row directly to the shared backtest runner. Model metrics do not choose a
configuration or checkpoint.

## Key takeaways and limitations

- The request records the label, estimator configuration, data reductions, and execution tier.
- Compatible estimators share fold preparation while retaining separate fitted states and result
  identities.
- The named validation population supports later analysis and backtesting without selecting on
  predictive metrics.
- Linear models restrict the conditional mean to the declared transformed feature basis;
  nonlinear structure requires a different family request.